# B1：IDM 跟驰模型参数标定
# B1: IDM Car-Following Model Calibration

## 一、实验原理 | Principle

### 1.1 什么是跟驰模型？
在真实交通中，驾驶员会根据与前车的**距离**和**速度差**来调整自己的加速度：
前车远 → 加速；前车近 → 减速；前车刹车 → 跟着刹车。
跟驰模型就是用数学公式描述这种驾驶行为。

### 1.2 IDM 模型公式
IDM（Intelligent Driver Model，智能驾驶员模型）由 Treiber 于 2000 年提出，
是交通仿真中最常用的跟驰模型。它的加速度公式为：

```
a = a_max · [ 1 - (v/v₀)⁴ - (s*/s)² ]
```

其中：
- `a_max`：最大加速度（车辆性能上限）
- `v`：自车当前速度，`v₀`：期望速度（空旷路想开多快）
- `s`：与前车的实际间距，`s*`：期望间距
- `s* = s₀ + v·T + v·Δv / (2√(a_max·b))`
  - `s₀`：静止最小间距（堵车时车贴多近）
  - `T`：期望时距（跟车时想保持几秒的距离）
  - `b`：舒适减速度（刹车想多柔和）
  - `Δv`：自车与前车的速度差

### 1.3 为什么要标定参数？
SUMO 自带一套默认 IDM 参数，但**真实驾驶员风格各异**：
有人跟车近、刹车晚（激进型），有人跟车远、刹车早（保守型）。
参数标定就是从**真实（或仿真）采集的跟驰数据**中，反推出最符合观测的参数值。
这是交通工程中的常用方法。

## 二、实验目的 | Objectives
1. 理解 IDM 模型的物理意义和每个参数的作用
2. 掌握用 SUMO + TraCI 采集跟驰数据的方法
3. 学会用最小二乘法标定模型参数
4. 用 RMSE 评价标定效果

## 三、学生任务 | Your Tasks
1. **运行**下面所有 cell，观察 SUMO 仿真和数据采集过程
2. **记录**标定出的 `a` 和 `b` 参数值，思考它们代表什么驾驶风格
3. **挑战**：把 `vType` 的 `accel` 改为 2.5（更激进的车），重新标定，对比参数变化
4. **思考题**：如果 `T`（期望时距）标定出来是 0.3s，这意味着什么？安全吗？

## 四、验证内容 | Verification
- [ ] SUMO 成功运行，采集到 >100 个数据点
- [ ] 标定算法收敛，RMSE < 2.0
- [ ] 标定出的参数在合理范围内（a: 0.5~3.0, b: 0.5~3.0）


In [ ]:
import numpy as np
print('✓ 导入成功')
print('本实验需要: numpy, scipy, traci, sumo')


## 五、IDM 模型定义 | IDM Model
下面是 IDM 加速度计算函数。注意 `max(gap, 0.1)` 防止除零——
这是数值仿真中的常见保护，间距不可能为负。


In [ ]:
def idm_acceleration(v, v0, gap, dv, a=1.0, b=1.5, s0=2.0, T=1.0):
    """IDM 加速度计算
    v: 自车速度, v0: 期望速度, gap: 间距, dv: 速度差（自车-前车）
    """
    s_star = s0 + v*T + v*dv/(2*np.sqrt(a*b))
    return a * (1 - (v/v0)**4 - (s_star/max(gap, 0.1))**2)

# 自检：静止跟驰（v=0）时加速度应为正（起步）
a_test = idm_acceleration(v=0, v0=25, gap=10, dv=0)
assert a_test > 0, 'IDM 函数异常!'
print(f'✓ IDM 自检通过：静止时加速度 = {a_test:.2f} m/s²')


## 六、从 SUMO 采集真实跟驰数据 | Collect Data from SUMO

### 常见报错说明 | Troubleshooting

| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `Vehicle 'lead' will not be able to depart...` | 车辆无法按指定速度出发（车道前方有慢车或位置冲突） | 降低 `departSpeed`，或错开 `depart` 时间 |
| `Vehicle 'lead' is not known` | TraCI 查询了还未出发的车辆 | 用 `try/except` 跳过，或检查 `getIDList()` |
| `SUMO_HOME is not set` | 环境变量未设置（不影响运行） | 忽略，或 `export SUMO_HOME=/usr/share/sumo` |
| 采集到 0 个数据点 | 车辆全程未成功运行 | 检查路网和路由文件是否正确生成 |


In [ ]:
import os
workdir = '/tmp/b1_sumo'
os.makedirs(workdir, exist_ok=True)
os.chdir(workdir)

# 生成路网（3x1 网格，每段 1000m）
ret = os.system('netgenerate --grid --grid.x-number=3 --grid.y-number=1 --grid.length=1000 --output-file=grid.net.xml > /dev/null 2>&1')
assert ret == 0, '路网生成失败！检查 netgenerate 是否可用'
print('✓ 路网生成成功')

# 路由：两辆车，前后相距 30m，先后出发（避免出发冲突）
# 注意：departPos 不要设在边线末端（如 1000），留余量
with open('routes.rou.xml', 'w') as f:
    f.write('''<routes>
    <vType id="car" carFollowModel="IDM" accel="1.5" decel="2.0" maxSpeed="25"/>
    <route id="r0" edges="A0B0 B0C0"/>
    <vehicle id="lead" type="car" route="r0" depart="0" departPos="200" departSpeed="12"/>
    <vehicle id="follow" type="car" route="r0" depart="0" departPos="170" departSpeed="12"/>
</routes>''')
with open('sumo.sumocfg', 'w') as f:
    f.write('<configuration><input><net-file value="grid.net.xml"/><route-files value="routes.rou.xml"/></input><time><step-length value="0.05"/></time></configuration>')
print('✓ 路由文件就绪')


In [ ]:
import traci
traci.start(['sumo', '-c', 'sumo.sumocfg', '--no-warnings', '--no-step-log'])
print('✓ SUMO 启动成功，开始采集...')

data = []  # (v, gap, dv, a_real)
for step in range(400):
    traci.simulationStep()
    ids = traci.vehicle.getIDList()
    if 'lead' not in ids or 'follow' not in ids:
        continue  # 车辆还未出发，跳过（避免 'not known' 报错）
    try:
        v = traci.vehicle.getSpeed('follow')
        v_lead = traci.vehicle.getSpeed('lead')
        x = traci.vehicle.getPosition('follow')[0]
        x_lead = traci.vehicle.getPosition('lead')[0]
        gap = x_lead - x - 4.5
        a = traci.vehicle.getAcceleration('follow')
        if gap > 0:  # 只记录有效数据
            data.append((v, gap, v - v_lead, a))
    except Exception as e:
        continue

traci.close()
print(f'✓ 采集 {len(data)} 个数据点（真 SUMO IDM 模型输出）')
assert len(data) > 100, f'数据点太少（{len(data)}），SUMO 可能未正常运行！检查上面的报错说明。'


## 七、参数标定（最小二乘）| Calibration

### 原理
我们有一批观测数据 `(v, gap, dv, a_real)`，想找到参数 `(a, b)` 使得
IDM 预测的加速度 `a_pred` 与真实加速度 `a_real` 的均方根误差（RMSE）最小。
用 `scipy.optimize.minimize` 做数值优化。


In [ ]:
from scipy.optimize import minimize
data = np.array(data)
v, gap, dv, a_real = data[:, 0], data[:, 1], data[:, 2], data[:, 3]

def loss(params):
    a_idm, b_idm = params
    a_pred = np.array([idm_acceleration(vi, 25, gi, dvi, a=a_idm, b=b_idm)
                       for vi, gi, dvi in zip(v, gap, dv)])
    return np.sqrt(np.mean((a_pred - a_real)**2))

print('标定中（可能需要 10-30 秒）...')
res = minimize(loss, x0=[1.0, 1.5], bounds=[(0.5, 3.0), (0.5, 3.0)])
a_opt, b_opt = res.x
rmse = res.fun
print(f'\n标定结果:')
print(f'  a (最大加速度) = {a_opt:.3f} m/s²')
print(f'  b (舒适减速度) = {b_opt:.3f} m/s²')
print(f'  RMSE = {rmse:.4f} m/s²')
print(f'\n解读：这辆车的驾驶风格是{"激进" if a_opt > 1.5 else "温和"}型')


## 八、评测 | Evaluation


In [ ]:
# 验证内容：
# 1. 数据量足够（>100 点）——已在采集 cell 中 assert
# 2. RMSE < 2.0 —— 标定精度达标
# 3. 参数在合理范围 —— 符合车辆物理
assert rmse < 2.0, f'RMSE {rmse:.2f} 超标！标定未收敛，检查数据质量。'
assert 0.5 <= a_opt <= 3.0, f'a={a_opt:.2f} 超出合理范围!'
assert 0.5 <= b_opt <= 3.0, f'b={b_opt:.2f} 超出合理范围!'
print('✅ 评测通过！IDM 参数标定成功')
print(f'   你的车：a={a_opt:.2f}, b={b_opt:.2f}, RMSE={rmse:.4f}')


## 九、🏆 挑战任务 | Challenges
1. **激进驾驶员**：把 `vType` 的 `accel` 改为 2.5，`decel` 改为 3.0，重新标定。参数如何变化？
2. **标定更多参数**：尝试同时标定 `T`（期望时距），观察 RMSE 能否进一步降低。
3. **思考题**：如果标定出的 `T=0.3s`，这意味着跟车时只保持 0.3 秒的距离。在 25m/s（90km/h）下这是多少米？安全吗？
